# TÓPICOS DE MATEMÁTICA PARA SISTEMA AUTÓNOMOS
## TRABALHO LETIVO - ENTREGA 1

**Nota 1:** Todos elementos do grupo são responsáveis por esta entrega
**Nota 2:** Todos elementos do grupo tomaram conhecimento das regras estabelecidas para o funcionamento da u.c. no Moodle do ISEP

## GRUPO 11
**N:** 1131518 **Nome:** Geovani Lopes  \
**N:** XXXXXXXX **Nome:** Hugo

<a id="conteudo"></a>
## Conteúdo
1. [Identificação dos objetivos e estruturação do problema](#sec1)
2. [Justificação da relevância das bibliotecas Python selecionadas](#sec2)
3. [Metodologia proposta para o desenvolvimento do projeto](#sec3)

<a id="sec1"></a>
## 1. Identificação dos objetivos e estruturação do problema
[De volta para "Conteúdo"](#conteudo)

### 1.1 Objetivo geral
Desenvolver um **dashboard web em Python/Dash** que reproduza (*replay*) missões multi-robô a partir de logs do tipo ROS, apresentando as trajetórias num **referencial global comum** e nos **referenciais locais** de cada robô, apesar de os logs terem referenciais e instantes de amostragem que não coincidem.

### 1.2 Objetivos específicos

| ID | Objetivo | Resultado verificável |
|----|----------|-----------------------|
| OE1 | **Global View**: todos os robôs num mesmo referencial mundo | Trajetórias, posição atual e identificação por cor de cada robô |
| OE2 | **Local View**: robô selecionado como origem | Trajetória no seu referencial, orientação (*heading*) e robôs vizinhos transformados para esse referencial |
| OE3 | **Mission Replay** | Slider temporal, *play/pause* e salto para um instante concreto |
| OE4 | **Configuração em YAML** | Caminhos dos logs, origem do referencial, parâmetros de sincronização, limiares de alerta e opções de exportação |
| OE5 | **Output Log / Output Configuration** | Log de eventos e estatísticas (CSV/JSON) e cópia da configuração efetivamente usada |
| **EX1** | **Exportação KML/GeoJSON** (extra) | Missão aberta no Google Earth e no QGIS |
| **EX2** | **Vista 3D em primeira pessoa** (extra) | Câmara 3D colocada no robô selecionado e orientada pelo seu *heading* |
| **EX3** | **Análise de proximidade e alertas** (extra) | Distância entre pares, CPA e alerta visual/registado em log |
| **EX4** | **Estatística geral por robô** (extra) | Painel com distância, profundidade, velocidades, acelerações e temperatura |

### 1.3 Estruturação do problema

**Entradas:** logs de missão (um por robô, tipo ROS: instante, posição, orientação e, quando existir, temperatura) e um ficheiro YAML de configuração.
**Saídas:** visualizações interativas (global, local, 3D), painel de estatísticas, alertas, ficheiros KML/GeoJSON e log de saída.

**Problema central.** Os robôs não partilham nem o referencial espacial nem a grelha temporal:

* **Referenciais diferentes:** posições em coordenadas geodésicas (LLA), ECEF ou locais (NED/ENU) com origens distintas.
* **Instantes de amostragem diferentes:** cada robô regista com a sua frequência, desfasamento (*offset*) e eventuais falhas, pelo que não existe um "frame" comum onde todos estejam simultaneamente definidos.

Para os representar em simultâneo é necessário (i) levar tudo para **um único referencial** e (ii) construir uma **grelha temporal comum** com os dados suavizados e interpolados.

| Sub-problema | Descrição | Ferramenta matemática |
|---|---|---|
| P1 | Leitura e normalização dos logs (unidades, nomes de campos, *timestamps*) | Parsing, validação |
| P2 | Referencial comum | Cadeia LLA → ECEF → NED |
| P3 | Sincronização e suavização temporal | Interpolação, filtragem, derivadas numéricas |
| P4 | Referencial local de um robô | Rotações 3D, transformação rígida |
| P5 | Replay interativo | Indexação temporal sobre a grelha comum |
| P6 | Extras (proximidade, estatísticas, exportação, vista 1.ª pessoa) | Geometria, cinemática, conversão inversa NED → LLA |

### 1.4 Modelo matemático

**Geodésico → ECEF (WGS84).** Com semieixo maior $a$, excentricidade $e$ e $N(\varphi)$ o raio de curvatura na vertical principal:

$$N=\frac{a}{\sqrt{1-e^{2}\sin^{2}\varphi}},\qquad
\begin{bmatrix}X\\Y\\Z\end{bmatrix}=
\begin{bmatrix}(N+h)\cos\varphi\cos\lambda\\(N+h)\cos\varphi\sin\lambda\\\left(N(1-e^{2})+h\right)\sin\varphi\end{bmatrix}$$

**ECEF → NED.** Dada a origem $(\varphi_0,\lambda_0,h_0)$ e o seu vetor ECEF $\mathbf p_0$:

$$\mathbf p_{NED}=R\,(\mathbf p_{ECEF}-\mathbf p_0),\qquad
R=\begin{bmatrix}
-\sin\varphi_0\cos\lambda_0 & -\sin\varphi_0\sin\lambda_0 & \cos\varphi_0\\
-\sin\lambda_0 & \cos\lambda_0 & 0\\
-\cos\varphi_0\cos\lambda_0 & -\cos\varphi_0\sin\lambda_0 & -\sin\varphi_0
\end{bmatrix}$$

**NED → referencial do robô $i$ (*body frame*, x à frente, y à direita, z para baixo).** Com ângulos de Euler $(\phi,\theta,\psi)$ e $R_b^n=R_z(\psi)R_y(\theta)R_x(\phi)$:

$$\mathbf p^{(i)}_{j}=\left(R_b^n\right)^{T}\,\left(\mathbf p_j-\mathbf p_i\right)$$

**NED → AER** (posição do robô $j$ vista do robô $i$, com $\Delta\mathbf p=(\Delta N,\Delta E,\Delta D)$):

$$az=\operatorname{atan2}(\Delta E,\Delta N),\qquad elev=\operatorname{atan2}\!\left(-\Delta D,\sqrt{\Delta N^{2}+\Delta E^{2}}\right),\qquad range=\lVert\Delta\mathbf p\rVert$$

### 1.5 Estratégia de sincronização e suavização
1. **Limpeza:** remover duplicados e *timestamps* não monótonos; detetar *outliers* (filtro de mediana/Hampel) e marcá-los.
2. **Janela comum:** $[t_{ini},t_{fim}]=[\max_i t^{(i)}_{primeiro},\ \min_i t^{(i)}_{último}]$, com possibilidade de usar a união e marcar as zonas sem dados.
3. **Grelha comum** com passo $\Delta t$ definido no YAML (por exemplo 0,1 s).
4. **Suavização por robô** (Savitzky–Golay ou filtro gaussiano) antes de derivar, para que velocidade e aceleração não amplifiquem o ruído.
5. **Reamostragem:** interpolação linear ou *spline* cúbica/PCHIP na posição; no *heading* usa-se `unwrap`, interpola-se e volta-se a normalizar para $[-\pi,\pi]$.
6. **Sem extrapolação** fora do intervalo de dados de cada robô: lacunas maiores que um limiar são marcadas como "sem dados" e não desenhadas.

### 1.6 Ideia adicional proposta (extras do grupo)

**EX1: Exportação KML/GeoJSON.** As trajetórias em NED são convertidas de volta para $(\varphi,\lambda,h)$ (NED → ECEF → LLA) e exportadas em **KML** (com `gx:Track`, que permite animação temporal no Google Earth) e **GeoJSON** (RFC 7946, coordenadas `[lon, lat, alt]`, com o instante em cada ponto para o controlo temporal do QGIS). Fecha o ciclo com ferramentas SIG reais e serve de validação cruzada das transformações.

**EX2: Vista 3D em primeira pessoa.** Para o robô selecionado, a câmara 3D é colocada na sua posição (com *offset* configurável) e orientada pelo vetor frontal $\mathbf f=R_b^n\,[1,0,0]^T$, com o vetor "cima" dado por $-z_{NED}$. Os restantes robôs aparecem como marcadores com identificação, e a câmara acompanha o slider de replay.

**EX3: Análise de proximidade e alertas.** Para cada par $(i,j)$ calcula-se $d_{ij}(t)=\lVert\mathbf p_i-\mathbf p_j\rVert$ e o **ponto de maior aproximação (CPA)**, com $\Delta\mathbf p=\mathbf p_j-\mathbf p_i$ e $\Delta\mathbf v=\mathbf v_j-\mathbf v_i$:

$$t^{*}=-\frac{\Delta\mathbf p\cdot\Delta\mathbf v}{\lVert\Delta\mathbf v\rVert^{2}},\qquad d_{CPA}=\lVert\Delta\mathbf p+\Delta\mathbf v\,t^{*}\rVert$$

São definidos dois níveis no YAML: **AVISO** ($d<d_{aviso}$) e **CRÍTICO** ($d<d_{crit}$, ou $d_{CPA}<d_{crit}$ dentro de um horizonte $T_h$), com histerese para evitar alertas intermitentes. Cada alerta é mostrado no dashboard (linha entre robôs e *banner*) e registado no log de saída.

**EX4: Estatística geral por robô.** Calculada sobre os dados suavizados e sincronizados:

| Grandeza | Definição |
|---|---|
| Distância total percorrida | $D=\sum_k\lVert\mathbf p_{k+1}-\mathbf p_k\rVert$ |
| Profundidade máxima | $\max_t(-h(t))$ (ou $\max_t z_{down}$ relativo à superfície, configurável) |
| Velocidade média | $D/(t_{fim}-t_{ini})$ |
| Velocidade atual | $\lVert\mathbf v(t)\rVert$, com $\mathbf v=d\mathbf p/dt$ |
| Aceleração atual | $\lVert\mathbf a(t)\rVert$, com $\mathbf a=d\mathbf v/dt$ |
| Aceleração angular atual | $\alpha=d\omega/dt$, com $\omega=d\psi/dt$ (*heading* com `unwrap`) |
| Temperatura atual | Valor do log no instante $t$; apresentada apenas se o campo existir |

<a id="sec2"></a>
## 2. Justificação da relevância das bibliotecas Python selecionadas
[De volta para "Conteúdo"](#conteudo)

A escolha segue três critérios: **(a)** cobrir diretamente um requisito do enunciado, **(b)** permitir um dashboard interativo único em Python, **(c)** manter o núcleo matemático explícito e verificável.

| Biblioteca | Papel no projeto | Justificação | Alternativa considerada |
|---|---|---|---|
| **Dash** | Aplicação web, *callbacks*, `dcc.Slider`, `dcc.Interval`, `dcc.Store` | Os *callbacks* dão controlo explícito sobre o estado do replay (slider, *play/pause*, salto de instante) e a vista local reage ao robô selecionado. É o *framework* indicado no enunciado. | Streamlit: o modelo de reexecução do *script* dificulta animação contínua e estado fino |
| **Plotly** | Gráficos 2D e 3D (`Scatter`, `Scatter3d`), câmara 3D | Interatividade nativa (zoom, *hover*) e integração direta com Dash. O `scene.camera` (*eye*, *center*, *up*) permite a vista em primeira pessoa (EX2). | Matplotlib: não é interativo na web; Bokeh/PyVista: integração com Dash mais limitada |
| **NumPy** | Núcleo numérico: matrizes de rotação, transformações, distâncias, CPA | Operações vetorizadas sobre séries temporais e matrizes $R$ em notação próxima da matemática da u.c. As transformações do núcleo são implementadas à mão para demonstrar domínio do modelo. | Listas Python: lentas e sem álgebra linear |
| **SciPy** | Interpolação (`interp1d`, `PchipInterpolator`, `CubicSpline`), filtros (`savgol_filter`), `Rotation` | Resolve o problema central de sincronização e suavização (P3), fornece derivadas numéricas robustas para velocidade e aceleração (EX4) e converte entre Euler e quaterniões. | Implementar splines à mão: sem ganho e mais propenso a erros |
| **pandas** | Leitura dos logs, alinhamento por tempo, estatísticas | `DataFrame` indexado por tempo, `read_csv`/`read_json`, `merge_asof` para análise de dessincronização e exportação do log de saída (OE5) em CSV. | CSV *parsing* manual: mais código e menos robusto |
| **pymap3d** | Referência para LLA/ECEF/NED/ENU/AER | Biblioteca de referência para as conversões do enunciado. Valida a implementação própria (testes de concordância) e faz a conversão inversa NED → LLA usada na exportação (EX1). | `pyproj`: forte em projeções cartográficas, menos direta para NED/AER |
| **PyYAML** | Leitura/escrita da configuração (OE4) e cópia da configuração efetiva (OE5) | Formato legível para o utilizador, com validação de campos obrigatórios e valores por omissão. | JSON: menos legível e sem comentários |
| **simplekml** | Geração de KML com `gx:Track` (EX1) | Cria KML válido com estilos e *tracks* temporais sem escrever XML à mão. | `lxml` manual: mais código para o mesmo resultado |
| **json** (biblioteca padrão) | Geração de GeoJSON (EX1) e logs de saída (OE5) | GeoJSON é JSON simples (RFC 7946), logo não exige dependência adicional. | `geojson`: dependência extra sem necessidade |
| **pytest** | Testes do núcleo matemático | Garante que as transformações, a sincronização e o CPA têm comportamento verificado (ver secção 3). | `unittest`: mais verboso |

**Nota sobre leitura de dados ROS:** os logs são tratados como ficheiros tabulares (CSV/JSON). Se for necessário ler *rosbags* reais, a biblioteca `rosbags` pode ser acrescentada apenas na camada de leitura, sem alterar o resto da aplicação.

<a id="sec3"></a>
## 3. Metodologia proposta para o desenvolvimento do projeto
[De volta para "Conteúdo"](#conteudo)

### 3.1 Princípios
* **Separação em camadas:** dados → núcleo matemático → visualização. O núcleo não depende do Dash e pode ser testado isoladamente.
* **Núcleo próprio validado por biblioteca de referência:** as transformações são implementadas em NumPy e verificadas contra `pymap3d`.
* **Dados sintéticos primeiro:** um gerador de missões com solução analítica conhecida permite validar antes de usar logs reais.
* **Desenvolvimento incremental:** cada fase termina com algo executável e testado.

### 3.2 Arquitetura proposta
```text
projeto/
├── config/mission.yaml          # configuração (OE4)
├── data/                        # logs de entrada (um por robô)
├── src/
│   ├── io_logs.py               # leitura e normalização (P1)
│   ├── frames.py                # LLA/ECEF/NED/AER e referencial local (P2, P4)
│   ├── sync.py                  # limpeza, suavização, grelha comum (P3)
│   ├── kinematics.py            # velocidade, aceleração, aceleração angular (EX4)
│   ├── proximity.py             # distâncias, CPA, alertas (EX3)
│   ├── export_geo.py            # KML e GeoJSON (EX1)
│   ├── outputs.py               # log de saída e configuração efetiva (OE5)
│   └── app/                     # Dash: layout e callbacks (OE1, OE2, OE3, EX2)
└── tests/                       # pytest
```

Exemplo de configuração prevista:
```yaml
origin: {lat: 41.1783, lon: -8.6080, h: 0.0}   # origem do referencial comum
sync: {dt: 0.1, interpolation: pchip, smoothing: savgol, window: 11, max_gap: 2.0}
proximity: {warn_distance: 10.0, critical_distance: 4.0, horizon: 5.0, hysteresis: 0.5}
depth: {mode: altitude_negative}
export: {formats: [kml, geojson], folder: out/}
```

### 3.3 Pipeline de processamento
```text
logs + YAML → leitura/normalização → conversão para NED comum → limpeza e suavização
→ reamostragem na grelha comum → cinemática e estatísticas → proximidade
→ Dash (Global, Local, 1.ª pessoa, Replay) → exportação (KML/GeoJSON) + log de saída
```

### 3.4 Fases e calendário

| Fase | Semana | Conteúdo | Entregável |
|---|---|---|---|
| F0 | 3 | Objetivos, bibliotecas e metodologia | Esta entrega (Entrega 1) |
| F1 | 4 | Leitura de logs, YAML, gerador sintético, transformações LLA/ECEF/NED/AER | `io_logs`, `frames` com testes |
| F2 | 5 | Sincronização e suavização; Global View | Todos os robôs em simultâneo no referencial comum |
| F3 | 6 | Local View, Mission Replay, estatísticas por robô | Replay completo e painel EX4 |
| F4 | 7 | Proximidade e alertas, vista 3D em primeira pessoa, exportação KML/GeoJSON | EX1, EX2 e EX3 funcionais |
| F5 | 8 | Testes finais, documentação, ensaio da defesa | Entrega 2 |

### 3.5 Verificação e validação

| Elemento | Teste |
|---|---|
| Transformações | Ida e volta LLA → ECEF → LLA com erro $<10^{-6}$ m; concordância com `pymap3d` |
| Referencial local | Robô e vizinho com posições conhecidas, resultado verificado à mão |
| Sincronização | Logs sintéticos com *offsets*, frequências diferentes e ruído, medindo o erro (RMSE) face à trajetória verdadeira |
| Cinemática | Trajetória circular de raio $r$ e velocidade angular $\omega$: $v=\omega r$, $a=\omega^2 r$, $\alpha=0$ |
| Proximidade | Cenário com CPA conhecido analiticamente; verificação da histerese |
| Exportação | Abrir os ficheiros no Google Earth e no QGIS e comparar com a Global View |
| Interface | Lista de verificação manual: slider, *play/pause*, salto, troca de robô, alertas |

### 3.6 Divisão de trabalho (sugestão)
* **Geovani:** `frames`, `sync`, `kinematics`, testes do núcleo matemático.
* **Hugo:** `io_logs`, interface Dash (Global, Local, Replay), exportação.
* **Em conjunto:** `proximity`, vista 3D em primeira pessoa, integração, documentação e preparação da defesa.

### 3.7 Riscos e mitigação

| Risco | Mitigação |
|---|---|
| Logs com formato ou unidades inesperados | Camada de leitura isolada, validação de campos e mensagens de erro claras |
| Ruído amplificado nas derivadas | Suavização antes de derivar; janela configurável |
| Desempenho do replay com muitos pontos | Decimação para visualização e `dcc.Store` com dados já reamostrados |
| Vista em 1.ª pessoa limitada pelo Plotly | Câmara via `scene.camera`; se necessário, simplificar para câmara com *offset* atrás do robô |
| Atraso nas últimas semanas | Extras ordenados por prioridade: EX4 → EX3 → EX1 → EX2 |

In [ ]:
# Verificação das dependências previstas para o projeto
import importlib.metadata as md

pacotes = ["numpy", "scipy", "pandas", "plotly", "dash", "pyyaml", "pymap3d", "simplekml", "pytest"]
for p in pacotes:
    try:
        print(f"{p:10s} {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p:10s} NÃO INSTALADO")